In [0]:
# 1. Define Auto Loader stream reading from folder containing sample.json
df_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.inferColumnTypes", "true")
         .option("cloudFiles.schemaLocation", f"{base_path}/_schema")
         .load(f"{base_path}/")
)

# 2. Add transformation
transformed_df = df_stream.withColumn("bonus", df_stream.salary * 0.10)

# 3. Process initial batch (John, Sarah, Mike, Emma)
query1 = (
    transformed_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{base_path}/_checkpoint")
        .trigger(availableNow=True)
        .toTable("employee_stream")
)
query1.awaitTermination()

# 4. Display baseline raw data output (4 rows)
display(spark.sql("SELECT * FROM employee_stream"))

department,employee_id,name,salary,_rescued_data,bonus
IT,101,John,60000,null,6000.0
HR,102,Sarah,55000,null,5500.0
IT,103,Mike,70000,null,7000.0
Finance,104,Emma,65000,null,6500.0


In [0]:
import json

simulated_data = [
    {"department": "Engineering", "employee_id": 106, "name": "Sophia", "salary": 110000},
    {"department": "Engineering", "employee_id": 107, "name": "Alex", "salary": 110000}
]

with open(f"{base_path}/simulated_data.json", "w") as f:
    for record in simulated_data:
        f.write(json.dumps(record) + "\n")

print("simulated_data.json created with Sophia & Alex!")

simulated_data.json created with Sophia & Alex!


In [0]:
# Re-run streaming write (picks up ONLY simulated_data.json)
query2 = (
    transformed_df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{base_path}/_checkpoint")
        .trigger(availableNow=True)
        .toTable("employee_stream")
)
query2.awaitTermination()

# Display updated table output (All 5 rows)
display(spark.sql("SELECT * FROM employee_stream"))

department,employee_id,name,salary,_rescued_data,bonus
IT,101,John,60000,null,6000.0
HR,102,Sarah,55000,null,5500.0
IT,103,Mike,70000,null,7000.0
Finance,104,Emma,65000,null,6500.0
Engineering,106,Sophia,110000,null,11000.0
Engineering,107,Alex,110000,null,11000.0
